# 04 · Calibration results: H1 and H2 (design Sections 3 and 8.1)

* **H1 (primary):** share of assets failing Kupiec at 5%, ours vs pooled split conformal; one-sided two-proportion
  z-test, Fisher exact check, Holm across the four levels at h = 21.
* **H2 (primary):** overall coverage, date-clustered test; yearly stability judged against pooled split conformal
  (amendment 1); coverage by year / asset class / VIX regime.
* `ours_vov` is the originally registered vol-of-vol method and `ours_sd` the amendment 1 version (mean/sd); both
  were replaced during development (amendments 1-2, see README).
* Pinball-loss Diebold-Mariano vs every comparator; robustness with HAR, GARCH and EWMA as base forecasters; other
  horizons (h = 5, 1) and the factor panels are secondary.
* Exploratory: conformal value-at-risk (Section 6.3).

In [ ]:
# ---------------------------------------------------------------- setup (run this cell first)
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CalVol_project"   # <- change if you placed the folder elsewhere
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements.txt"], check=True)
else:
    PROJECT_DIR = os.path.abspath("..")
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

import json
import numpy as np, pandas as pd
from IPython.display import display, Image, Markdown
from calvol.utils import load_config, get_paths, check_gate
from calvol import pipeline as pp
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)

cfg = load_config(PROJECT_DIR)          # mode / sign-off / smoke test are set in config/config.yaml (project: ...)
MODE = check_gate(cfg)                  # refuses a holdout run without sign-off and a frozen commit
P = get_paths(cfg)
PER = pp.eval_split(cfg)
print(f"MODE = {MODE.upper()}  |  evaluation period = {PER}  |  smoke test = {cfg['project']['smoke_test']}")
print("Results folder:", P.results)

def show(name):
    f = P.figures / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=760))
    else:
        print("(figure not produced:", name, ")")

def table(name, round_=4):
    f = P.tables / f"{name}.csv"
    if f.exists():
        display(pd.read_csv(f).round(round_))
    else:
        print("(table not produced:", name, ")")

In [ ]:
JOBS = [("assets", h) for h in cfg["panels"]["assets"]["horizons"]] + [(p, 21) for p in pp.FACTOR_PANELS]
res = {}
for name, h in JOBS:
    res[(name, h)] = pp.stage_calibration(cfg, name, h)
    print("done", name, h)

## Primary: assets, h = 21

In [ ]:
h = cfg["evaluation"]["primary_horizon"]
display(Markdown("**H1** (primary row: base dl, comparator pooled_split; `p_holm_primary`)"))
table(f"assets_h{h}_H1")
display(Markdown("**H2**")); table(f"assets_h{h}_H2")
show(f"fig_assets_h{h}_kupiec"); show(f"fig_assets_h{h}_per_asset_coverage"); show(f"fig_assets_h{h}_coverage_by_year")

## Full calibration table (every base x method x level) and pinball DM

In [ ]:
display(pd.read_csv(P.tables / f"assets_h{h}_calibration_all.csv").round(4))
table(f"assets_h{h}_pinball_dm")

## Coverage of our bounds by year, asset class and VIX regime

In [ ]:
g = pd.read_csv(P.tables / f"assets_h{h}_coverage_by_group.csv")
for k, d in g.groupby("grouping"):
    display(Markdown(f"**{k}**")); display(d.pivot(index="group", columns="level", values="coverage").round(3))

## Secondary: other horizons and the factor panels

In [ ]:
for name, hh in JOBS:
    if (name, hh) == ("assets", h):
        continue
    display(Markdown(f"### {name}, h = {hh}"))
    table(f"{name}_h{hh}_H1"); table(f"{name}_h{hh}_H2")
    show(f"fig_{name}_h{hh}_kupiec")

## Exploratory: conformal value-at-risk (no hypothesis attached)

In [ ]:
for hh in cfg["panels"]["assets"]["horizons"]:
    display(Markdown(f"**assets, h = {hh}**"))
    display(pp.stage_conformal_var(cfg, "assets", hh).round(4))